# S6E10 — 22 Auxiliary features, second version: multiclass expectation + surprise

> **Pipeline position:** 13–14 aux features v1 (rejected) → **22 aux v2** → 11 stack

| | |
|---|---|
| **Input** | `data/raw/train.csv`, `test.csv` (no labels for the features); `X_rich_*`, `y_train` for the RealMLP test; reference checkpoint of `12` |
| **Output** | `data/processed/X_auxmc_train.parquet`, `X_auxmc_test.parquet` (32 columns); `preds/realmlp_rich_auxmc_folds/` (fold 0) |
| **Runtime** | features ≈ 20–30 min (GPU XGBoost, once); RealMLP fold 0 ≈ 2 min |
| **Result** | see *Results & decisions* |

**Why a second version.** `13`–`14` predicted each column from the other 20 with a **regression** and gave RealMLP the expected value and the deviation from it: no gain. A public series on this competition (goodpjw2008, Part 4) reports +0.00017 for RealMLP from auxiliary-task features built differently: a **multiclass** model per rating, and besides the expected value Σ k·P(k), the **probability the model gives to the answer the passenger actually gave** (`aux_p`). The deviation of v1 says *how far* an answer is from the typical one; `aux_p` says *how unusual* it is — a rating of 5 where the model expected {4: 0.5, 5: 0.45} is ordinary, a 5 where it expected {1: 0.9} is a surprise, even if the expected values were similar. This notebook implements that idea ourselves and tests it with the same paired fold-0 protocol as `14`.

**Columns (16):** the 13 ratings as up to 6 classes (the values observed, 0–5) → `aux_ev`, `aux_p`; travel type, customer type and class (Business vs not) as binary → `aux_ev` = P(1), `aux_p` = P(actual value).

**No labels, no self-leakage:** train+test are split into 5 parts unrelated to the competition folds; each part is predicted by a model trained on the other 4, so a row's own value never helps predict it. The satisfaction label is not read in section 1.

> **Glossary:** *Multiclass model* = predicts a probability for every possible value (here 0–5). *Expected value* = Σ k·P(k). *Surprise (`aux_p`)* = P(the value the row actually has).

In [1]:
import torch          # first import (project convention)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import re, time, gc, warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RAW = next(p for p in [Path('data/raw'), Path('../data/raw')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed')] if (p / 'X_rich_train.parquet').exists())
PREDS_DIR = next(p for p in [Path('preds'), Path('../preds')] if p.exists())
PARTS = PROC / 'auxmc_parts'; PARTS.mkdir(exist_ok=True)
train = pd.read_csv(RAW / 'train.csv'); test = pd.read_csv(RAW / 'test.csv')
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in test.columns if c not in CATS + NUMS + ['id']]
BASE = NUMS + RATINGS + CATS
full = pd.concat([train[BASE], test[BASE]], ignore_index=True)     # leakage guard: the label column is not taken
for c in CATS: full[c] = full[c].astype('category')
clean = lambda s: re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()
TARGETS = {r: ('multi', full[r].astype(int).values) for r in RATINGS}
TARGETS['Type of Travel'] = ('bin', (full['Type of Travel'] == 'Business travel').astype(int).values)
TARGETS['Customer Type'] = ('bin', (full['Customer Type'] == 'Loyal Customer').astype(int).values)
TARGETS['Class'] = ('bin', (full['Class'] == 'Business').astype(int).values)
AUX_FOLDS = list(KFold(5, shuffle=True, random_state=42).split(np.zeros(len(full))))
print(DEVICE, full.shape, '| rating values:', sorted(full[RATINGS[0]].unique()))

cuda (999479, 21) | rating values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 1. Cross-predicted multiclass models (resumable per column)

In [2]:
PARAMS = dict(n_estimators=300, learning_rate=0.1, max_depth=7, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
              tree_method='hist', device=DEVICE, enable_categorical=True, random_state=42, verbosity=0)
EV, PA = {}, {}
for col, (kind, t) in TARGETS.items():
    f = PARTS / f'{clean(col)}.npz'
    if f.exists():
        z = np.load(f); EV[col], PA[col] = z['ev'], z['p']; print(f'{col:35s} loaded'); continue
    t0 = time.time(); Xc = full.drop(columns=[col]); ev = np.zeros(len(full), np.float32); pa = np.zeros(len(full), np.float32)
    for a, b in AUX_FOLDS:
        if kind == 'multi':
            vals = np.unique(t); code_ = np.searchsorted(vals, t)   # classes re-coded 0..K-1 (e.g. Baggage handling has no 0)
            m = xgb.XGBClassifier(objective='multi:softprob', **PARAMS).fit(Xc.iloc[a], code_[a])
            P = m.predict_proba(Xc.iloc[b])                       # columns = the K observed values, in increasing order
            ev[b] = P @ vals; pa[b] = P[np.arange(len(b)), code_[b]]
        else:
            m = xgb.XGBClassifier(objective='binary:logistic', **PARAMS).fit(Xc.iloc[a], t[a])
            p1 = m.predict_proba(Xc.iloc[b])[:, 1]; ev[b] = p1; pa[b] = np.where(t[b] == 1, p1, 1 - p1)
    np.savez(f, ev=ev, p=pa); EV[col], PA[col] = ev, pa
    print(f'{col:35s} mean P(actual) {pa.mean():.3f} | share P(actual) < 0.1: {(pa < 0.1).mean():.3f} | {time.time() - t0:.0f}s', flush=True)

A = pd.DataFrame({**{f'auxmc_ev_{clean(c)}': EV[c] for c in TARGETS}, **{f'auxmc_p_{clean(c)}': PA[c] for c in TARGETS}}).astype('float32')
n_tr = len(train)
gap = (A.iloc[:n_tr].mean() - A.iloc[n_tr:].mean()).abs() / A.std()
print('columns:', A.shape[1], '| largest train/test mean gap (std units):', round(float(gap.max()), 4))
A.iloc[:n_tr].reset_index(drop=True).to_parquet(PROC / 'X_auxmc_train.parquet'); A.iloc[n_tr:].reset_index(drop=True).to_parquet(PROC / 'X_auxmc_test.parquet')

Inflight wifi service               mean P(actual) 0.844 | share P(actual) < 0.1: 0.057 | 53s


Departure/Arrival time convenient   mean P(actual) 0.697 | share P(actual) < 0.1: 0.055 | 56s


Ease of Online booking              mean P(actual) 0.854 | share P(actual) < 0.1: 0.060 | 51s


Gate location                       mean P(actual) 0.617 | share P(actual) < 0.1: 0.053 | 53s


Food and drink                      mean P(actual) 0.642 | share P(actual) < 0.1: 0.023 | 51s


...

Cleanliness                         mean P(actual) 0.673 | share P(actual) < 0.1: 0.025 | 50s


Type of Travel                      mean P(actual) 0.985 | share P(actual) < 0.1: 0.002 | 11s


Customer Type                       mean P(actual) 0.986 | share P(actual) < 0.1: 0.001 | 11s


Class                               mean P(actual) 0.936 | share P(actual) < 0.1: 0.008 | 11s


columns: 32 | largest train/test mean gap (std units): 0.004


## 2. RealMLP-rich + aux v2, fold 0 (paired with the reference of `12`: 0.96138)
Same recipe, 4 networks, seed 0, same machine; the only change is the 32 extra columns. Variant `p` adds only the 16 surprise columns (the part that is new compared with `14`).

In [3]:
from sklearn.preprocessing import TargetEncoder
from pytabkit import RealMLP_TD_Classifier
X0 = pd.read_parquet(PROC / 'X_rich_train.parquet'); X0_test = pd.read_parquet(PROC / 'X_rich_test.parquet')
y = pd.read_parquet(PROC / 'y_train.parquet')['y'].astype(int).values
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X0, y)); tr0, va0 = FOLDS[0]
Atr, Ate = pd.read_parquet(PROC / 'X_auxmc_train.parquet'), pd.read_parquet(PROC / 'X_auxmc_test.parquet')
RATINGS_L = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
             'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
             'checkin_service','cleanliness']
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS_L}}
KEYS = [c for c in X0.columns if c.startswith('key_')]
TE_SOURCES = ['flight_distance', 'age', 'fd_div10', 'fd_mod1000'] + KEYS
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)
FULLX = pd.concat([X0, X0_test], ignore_index=True)
LEVELS = {t: sorted(FULLX[s].astype(int).unique()) for t, s in TWINS.items()}
PARAMS_NN = dict(n_epochs=4, n_ens=4, batch_size=256, use_early_stopping=False, val_metric_name='1-auc_ovr', device=DEVICE, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15', first_layer_lr_factor=0.25, embedding_size=5,
    max_one_hot_cat_size=18, hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t', plr_hidden_1=16, plr_hidden_2=8,
    plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33, use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False,
    bias_init_mode='neg-uniform-dynamic-2', tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'])

def nn_frame(D):
    D = D.drop(columns=['orig_prob'] + KEYS).copy()
    for t, s in TWINS.items(): D[t] = pd.Categorical(D[s].astype(int), categories=LEVELS[t])
    return D
def add_te(Xtr, ytr, others):
    # in-fold target encoding: fitted on the fold's TRAINING part only (inner cv=5), applied to validation/test
    te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42); cols = [f'te_{c}' for c in TE_SOURCES]
    tr_te = pd.DataFrame(te.fit_transform(Xtr[TE_SOURCES].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
    outs = [pd.DataFrame(te.transform(o[TE_SOURCES].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
    return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]

P = {}
for tag, cols in [('all', list(Atr.columns)), ('p', [c for c in Atr.columns if c.startswith('auxmc_p_')])]:
    d = PREDS_DIR / f'realmlp_rich_auxmc{"" if tag == "all" else "_p"}_folds'; f = d / 'oof_0.npy'
    if f.exists(): P[tag] = np.load(f); print(tag, 'loaded'); continue
    X = pd.concat([X0, Atr[cols]], axis=1); Xt = pd.concat([X0_test, Ate[cols]], axis=1)
    t0 = time.time()
    Xtr, Xva, Xte = add_te(X.iloc[tr0], y[tr0], [X.iloc[va0], Xt]); Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
    m = RealMLP_TD_Classifier(**{**PARAMS_NN, 'random_state': 0})
    m.fit(Xtr, y[tr0], X_val=Xva, y_val=y[va0], cat_col_names=CAT_COLS)      # X_val for logging only (no early stopping)
    P[tag] = m.predict_proba(Xva)[:, 1]; d.mkdir(exist_ok=True); np.save(f, P[tag]); np.save(d / 'test_0.npy', m.predict_proba(Xte)[:, 1])
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f'{tag}: {(time.time() - t0) / 60:.1f} min', flush=True)
a_ref = roc_auc_score(y[va0], np.load(PREDS_DIR / 'realmlp_rich_refgpu_folds' / 'oof_0.npy'))
for tag in P: print(f'RealMLP-rich + aux v2 ({tag:3s}): fold 0 {roc_auc_score(y[va0], P[tag]):.5f} | reference {a_ref:.5f} | diff {roc_auc_score(y[va0], P[tag]) - a_ref:+.5f}')

W1010 07:16:49.764000 19248 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


p: 1.4 min


RealMLP-rich + aux v2 (all): fold 0 0.96143 | reference 0.96138 | diff +0.00006
RealMLP-rich + aux v2 (p  ): fold 0 0.96137 | reference 0.96138 | diff -0.00000


## 3. Mini-stack on the rows of fold 0

In [4]:
from sklearn.linear_model import LogisticRegression
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
as_logit = lambda o: lg(o) if (o.min() >= 0 and o.max() <= 1) else o
NAMES = {'cat_v1': 'cat_v1', 'cat_v2': 'cat_v2', 'xgb': 'xgb', 'lgbm_tuned': 'lgbm_tuned_final', 'mlp': 'mlp', 'realmlp': 'realmlp_avg',
         'realmlp_rich': 'realmlp_rich_avg', 'tabpfn': 'tabpfn_avg', 'tabpfn_raw': 'tabpfn_raw_avg', 'tabicl': 'tabicl'}
B = np.column_stack([as_logit(np.load(PREDS_DIR / f'oof_{s}.npy').astype(float))[va0] for s in NAMES.values()]); yv = y[va0]
inner = list(StratifiedKFold(5, shuffle=True, random_state=42).split(B, yv))
score = lambda M: np.array([roc_auc_score(yv[b], M[b] @ LogisticRegression(C=10.0, max_iter=500).fit(M[a], yv[a]).coef_[0]) for a, b in inner])
base = score(B)
for tag in P:
    s_ = score(np.hstack([B, lg(P[tag])[:, None]])); d_ = s_ - base
    print(f'+ RealMLP aux v2 ({tag}): mini-stack {s_.mean():.5f} vs {base.mean():.5f} | diff {d_.mean():+.5f} | positive {(d_ > 0).sum()}/5')

+ RealMLP aux v2 (all): mini-stack 0.96193 vs 0.96193 | diff +0.00001 | positive 3/5


+ RealMLP aux v2 (p): mini-stack 0.96193 vs 0.96193 | diff +0.00000 | positive 3/5


## Results & decisions

**Aux v2 features** (16 multiclass / binary XGBoost models, in-fold cross-fitted, ≈ 14 min on GPU): reconstruction is far from trivial for most ratings (mean P(actual) 0.29–0.92), near-perfect for the three categoricals (≈ 0.99). Train/test distributions agree (largest mean gap 0.004 std).

| Fold 0 | AUC | vs reference 0.96138 | Mini-stack (10 members + it) | Positive inner folds |
|---|---|---|---|---|
| RealMLP-rich + aux v2, all 32 columns | 0.96143 | +0.00006 | +0.00001 | 3/5 |
| RealMLP-rich + aux v2, `p` only (16) | 0.96137 | −0.00000 | +0.00000 | 3/5 |

**Decision: rejected.** +0.00006 on one fold is below the noise threshold (+0.0002), and the mini-stack gain is zero. Together with `13`–`14` (aux v1, +0.000005) this closes the *auxiliary reconstruction* idea: how surprising a passenger's rating is given the rest of the row is already captured by the network itself. No further folds are run. The stack stays at 10 members.


## Glossary
* **Multiclass auxiliary model** — predicts P(value = k) for k = 0…5 of one rating from the other 20 columns.
* **`auxmc_ev_*`** — expected value Σ k·P(k). **`auxmc_p_*`** — probability of the value actually given (low = surprising answer).
* **Paired benchmark** — the reference of `12`, trained under identical conditions.